## 1. Crear base de datos y cargar datos

In [ ]:
import sqlite3
from pathlib import Path

conn = sqlite3.connect("db.sqlite")
cur = conn.cursor()

for fname in ["schema.sql", "grados_familias_ciclos_modulos.sql", "convalidaciones.sql"]:
    sql = Path(fname).read_text(encoding="utf-8")
    cur.executescript(sql)

conn.commit()
print("Base de datos cargada correctamente.")

## 2. Módulos por ciclo formativo (con familia y grado)

In [ ]:
import pandas as pd

query = """
SELECT
    g.nombre      AS grado,
    f.codigo      AS cod_familia,
    f.nombre      AS familia,
    c.nombre      AS ciclo,
    m.id_oficial  AS cod_modulo,
    m.nombre      AS modulo
FROM modulos m
JOIN ciclos   c ON m.id_ciclo  = c.id
JOIN familias f ON c.id_familia = f.id
JOIN grados   g ON c.id_grado   = g.id
ORDER BY g.id, f.codigo, c.nombre, m.id_oficial;
"""

df = pd.read_sql_query(query, conn)
print(f"Total filas: {len(df)}")


## 3. Filtrar por ciclo concreto

In [ ]:
# Cambia este valor para filtrar por cualquier ciclo
CICLO = "Técnico en Gestión Administrativa"

df_ciclo = df[df["ciclo"] == CICLO].reset_index(drop=True)
print(f"Ciclo: {CICLO}")
print(f"Familia: {df_ciclo['familia'].iloc[0]}  |  Grado: {df_ciclo['grado'].iloc[0]}")
print(f"Nº módulos: {len(df_ciclo)}")
df_ciclo[["cod_modulo", "modulo"]]

## 4. Resumen: número de módulos por ciclo

In [ ]:
resumen = (
    df.groupby(["grado", "cod_familia", "familia", "ciclo"])
    .size()
    .reset_index(name="num_modulos")
)
resumen

## 5. Análisis de convalidaciones: módulos requeridos

In [ ]:
query_conv = """
SELECT
    c.id AS id_convalidacion,
    f.nombre AS familia_destino,
    cic.nombre AS ciclo_destino,
    m.id_oficial AS cod_modulo_destino,
    m.nombre AS modulo_destino,
    COUNT(co.id_modulo) AS num_modulos_origen
FROM convalidacion c
JOIN modulos m ON c.id_modulo_destino = m.id
JOIN ciclos cic ON m.id_ciclo = cic.id
JOIN familias f ON cic.id_familia = f.id
LEFT JOIN convalidacion_origen co ON c.id = co.conv_id
GROUP BY c.id
ORDER BY familia_destino, ciclo_destino, modulo_destino;
"""

df_conv = pd.read_sql_query(query_conv, conn)


## 6. Cerrar conexión

In [ ]:
conn.close()
print("Conexión cerrada.")